# 2회차 · LLM 성능평가 및 프로젝트 모델 선정

**실습 환경:** RunPod A100 · VS Code Remote-SSH · Jupyter Notebook · Hugging Face  
**평가 대상:** `google/gemma-3-4b-it` BF16, `Qwen/Qwen3-8B` BF16 (1회차와 동일 모델)  
**목표:** 동일한 5문항에서 Latency / Tokens/sec / Peak VRAM / 형식 준수율 / Raw JSON 유효성을 측정하고 모델 선정 근거를 기록합니다.

> 아래 출력값은 실행 시 측정합니다. 과거 예시 수치를 실제 측정값으로 간주하지 않습니다. Qwen Thinking 모드 OFF. Gemma는 Hugging Face 라이선스 승인 및 인증 필요.

## STEP 1. GPU 연결 및 실행 환경 점검

In [ ]:
import sys, subprocess, torch
print("커널:",sys.executable)
print("CUDA:",torch.cuda.is_available())
assert torch.cuda.is_available(), "RunPod GPU의 Python 커널을 선택하세요."
print("GPU:",torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"],check=True)

## STEP 2. 패키지 설치

In [ ]:
%pip install -U "transformers>=4.53,<5" accelerate pandas pillow sentencepiece huggingface_hub

## STEP 3. 공통 평가 데이터와 결과 폴더

In [ ]:
import os, re, gc, json, time
from pathlib import Path
import pandas as pd
from IPython.display import display

os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
Path(os.environ["HF_HOME"]).mkdir(parents=True,exist_ok=True)
RESULTS_DIR=Path("/workspace/results")
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
MAX_NEW_TOKENS=200
EVALUATION_DATA=[
    {"id":1,"category":"설명","prompt":"생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."},
    {"id":2,"category":"요약","prompt":"다음 내용을 한 문장으로 요약해줘. GPU는 병렬 연산에 특화되어 있고 딥러닝의 대규모 행렬 계산에 쓰인다."},
    {"id":3,"category":"지시준수","prompt":"오픈소스 LLM의 장점을 정확히 3개만 1. 2. 3. 형식으로 작성해줘."},
    {"id":4,"category":"구조화 출력","prompt":"Qwen과 Gemma를 비교하는 정보를 model, developer, usage 세 키를 가진 JSON 객체로만 출력해줘. Markdown 금지."},
    {"id":5,"category":"분류","prompt":"'GPU를 사용하니 추론 속도가 향상되었다.'를 긍정, 부정, 중립 중 하나로만 분류해줘."},
]
display(pd.DataFrame(EVALUATION_DATA))
# STEP 6의 Gemma 인증에 사용: CLI 로그인 토큰을 자동으로 읽음
def runpod_hf_token():
    """Return the current HF token without printing it or saving it in notebook source."""
    from pathlib import Path
    import os
    from huggingface_hub import get_token

    # The user authenticated in the RunPod shell (seen at this CLI token location).
    paths = [
        Path('/workspace/.cache/huggingface/token'),
        Path('/workspace/hf-cache/token'),
        Path.home() / '.cache' / 'huggingface' / 'token',
    ]
    for path in paths:
        if path.is_file():
            token = path.read_text(encoding='utf-8').strip()
            if token:
                return token
    token = os.environ.get('HF_TOKEN') or get_token()
    if not token:
        raise RuntimeError(
            'Hugging Face 토큰을 찾지 못했습니다. RunPod 터미널에서 hf auth login 후 다시 실행하세요.'
        )
    return token


,id,category,prompt
0,1,설명,생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘.
1,2,요약,다음 내용을 한 문장으로 요약해줘. GPU는 병렬 연산에 특화되어 있고 딥러닝의 대...
2,3,지시준수,오픈소스 LLM의 장점을 정확히 3개만 1. 2. 3. 형식으로 작성해줘.
3,4,구조화 출력,"Qwen과 Gemma를 비교하는 정보를 model, developer, usage ..."
4,5,분류,"'GPU를 사용하니 추론 속도가 향상되었다.'를 긍정, 부정, 중립 중 하나로만 분..."


## STEP 4. 평가 공통 함수 — GPU 시간·VRAM 측정
두 모델에서 동일한 생성 설정을 쓰되 채팅 템플릿은 모델별로 다르게 적용합니다. **Tokens/sec는 입력 처리까지 포함한 전체 소요시간 기준의 간이 생성량 지표**입니다.

In [ ]:
def prepare_inputs(prompt, adapter, model, *, gemma=False):
    if gemma:
        messages=[{"role":"user","content":[{"type":"text","text":prompt}]}]
        inputs=adapter.apply_chat_template(messages,add_generation_prompt=True,
                tokenize=True,return_dict=True,return_tensors="pt").to(model.device)
        decoder=adapter.tokenizer
    else:
        messages=[{"role":"user","content":prompt}]
        text=adapter.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,
                                         enable_thinking=False)
        inputs=adapter(text,return_tensors="pt").to(model.device)
        decoder=adapter
    return inputs,decoder

def evaluate_one(item, model, adapter, *, model_name, gemma=False):
    inputs,decoder=prepare_inputs(item["prompt"],adapter,model,gemma=gemma)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter()
    with torch.inference_mode():
        output=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
                              pad_token_id=decoder.eos_token_id)
    torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    new_ids=output[0][inputs["input_ids"].shape[-1]:]
    response=decoder.decode(new_ids,skip_special_tokens=True).strip()
    return {**item,"model":model_name,"response":response,
        "generated_tokens":int(new_ids.numel()),"latency_sec":round(elapsed,3),
        "tokens_per_sec":round(new_ids.numel()/elapsed,2) if elapsed else None,
        "peak_vram_gb":round(torch.cuda.max_memory_allocated()/1024**3,2)}

def run_benchmark(model,adapter,*,model_name,gemma=False):
    rows=[]
    for item in EVALUATION_DATA:
        measured=evaluate_one(item,model,adapter,model_name=model_name,gemma=gemma)
        rows.append(measured)
        print(f"[{model_name}] {item['category']}: {measured['latency_sec']}초, {measured['tokens_per_sec']} tok/s")
    return pd.DataFrame(rows)

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print("정리 후 할당 VRAM(GB):",round(torch.cuda.memory_allocated()/1024**3,2))

## STEP 5. 출력 형식 준수 및 JSON 검사 함수
이 수업의 `instruction_pass`는 **자동 규칙 기반 형식 준수**이며 실제 정답 정확도(Accuracy)가 아닙니다.

In [ ]:
def clean_fenced_json(raw):
    text=raw.strip()
    if text.startswith("```"):
        text=re.sub(r"^```(?:json)?\s*", "",text,flags=re.I)
        text=re.sub(r"\s*```$", "",text)
    return text.strip()

def json_parses(text):
    try:
        obj=json.loads(text)
        return True,obj
    except json.JSONDecodeError:
        return False,None

def count_sentences(text):
    return len([part for part in re.split(r"[.!?]+(?:\s+|$)",text.strip()) if part.strip()])

def format_check(row):
    text=row["response"].strip()
    kind=row["category"]
    raw_ok, parsed=json_parses(text)
    clean_ok,_=json_parses(clean_fenced_json(text))
    if kind=="설명":
        passed=count_sentences(text)==3
    elif kind=="요약":
        passed=count_sentences(text)==1
    elif kind=="지시준수":
        passed=len(re.findall(r"(?m)^\s*[123][.)]\s+",text))==3
    elif kind=="구조화 출력":
        passed=raw_ok and isinstance(parsed,dict) and {"model","developer","usage"}.issubset(parsed)
    elif kind=="분류":
        passed=text in ("긍정","부정","중립")
    else:
        passed=False
    return pd.Series({"instruction_pass":bool(passed),
        "raw_json_valid":bool(raw_ok) if kind=="구조화 출력" else None,
        "cleaned_json_valid":bool(clean_ok) if kind=="구조화 출력" else None})

def enrich_quality(df):
    result=pd.concat([df.reset_index(drop=True),df.apply(format_check,axis=1)],axis=1)
    return result

## STEP 6. Gemma 3 4B BF16 모델 로드와 5문항 평가

In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

GEMMA_ID="google/gemma-3-4b-it"
# 토큰 경로: /workspace/.cache/huggingface/token 등
# `hf download google/gemma-3-4b-it config.json` 성공 후에도 Notebook이 401일 때 해결
hf_token=runpod_hf_token()
processor=AutoProcessor.from_pretrained(GEMMA_ID,token=hf_token)
model=Gemma3ForConditionalGeneration.from_pretrained(
    GEMMA_ID,token=hf_token,dtype=torch.bfloat16,device_map="auto"
).eval()
del hf_token
print("Gemma 로딩 완료:",torch.cuda.get_device_name(0))
gemma_df=enrich_quality(run_benchmark(model,processor,model_name="Gemma-3-4B-IT",gemma=True))
display(gemma_df[["category","generated_tokens","latency_sec","tokens_per_sec","peak_vram_gb","instruction_pass"]])

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Gemma 로딩 완료: NVIDIA A100 80GB PCIe
[Gemma-3-4B-IT] 설명: 6.71초, 20.42 tok/s
[Gemma-3-4B-IT] 요약: 1.54초, 17.53 tok/s
[Gemma-3-4B-IT] 지시준수: 9.14초, 21.88 tok/s
[Gemma-3-4B-IT] 구조화 출력: 8.555초, 23.38 tok/s
[Gemma-3-4B-IT] 분류: 0.225초, 13.36 tok/s


,category,generated_tokens,latency_sec,tokens_per_sec,peak_vram_gb,instruction_pass
0,설명,137,6.710,20.42,8.04,True
1,요약,27,1.540,17.53,8.03,True
2,지시준수,200,9.140,21.88,8.05,True
3,구조화 출력,200,8.555,23.38,8.05,False
4,분류,3,0.225,13.36,8.03,True


## STEP 7. Gemma GPU 메모리 정리

In [ ]:
del model, processor
clear_gpu()

정리 후 할당 VRAM(GB): 0.01


## STEP 8. Qwen3-8B BF16 모델 로드 및 같은 5문항 평가

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_ID="Qwen/Qwen3-8B"
tokenizer=AutoTokenizer.from_pretrained(QWEN_ID)
model=AutoModelForCausalLM.from_pretrained(
    QWEN_ID,dtype=torch.bfloat16,device_map="auto"
).eval()
print("Qwen 로딩 완료:",torch.cuda.get_device_name(0))
qwen_df=enrich_quality(run_benchmark(model,tokenizer,model_name="Qwen3-8B",gemma=False))
display(qwen_df[["category","generated_tokens","latency_sec","tokens_per_sec","peak_vram_gb","instruction_pass"]])

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen 로딩 완료: NVIDIA A100 80GB PCIe
[Qwen3-8B] 설명: 3.045초, 29.56 tok/s
[Qwen3-8B] 요약: 0.949초, 29.51 tok/s
[Qwen3-8B] 지시준수: 5.723초, 31.1 tok/s
[Qwen3-8B] 구조화 출력: 2.122초, 33.93 tok/s
[Qwen3-8B] 분류: 0.106초, 28.28 tok/s


,category,generated_tokens,latency_sec,tokens_per_sec,peak_vram_gb,instruction_pass
0,설명,90,3.045,29.56,15.28,True
1,요약,28,0.949,29.51,15.28,True
2,지시준수,178,5.723,31.10,15.30,True
3,구조화 출력,72,2.122,33.93,15.28,False
4,분류,3,0.106,28.28,15.28,True


## STEP 9. Qwen 메모리 정리

In [ ]:
del model,tokenizer
clear_gpu()

정리 후 할당 VRAM(GB): 0.01


## STEP 10. 두 모델의 결과 비교 및 프로젝트 모델 선정
모든 수치는 이번 RunPod 실행 시 측정한 값입니다. 응답 본문과 함께 비교하세요.

In [ ]:
compare_df=pd.concat([gemma_df,qwen_df],ignore_index=True)
selection_summary=(compare_df.groupby("model")
    .agg(avg_latency_sec=("latency_sec","mean"),
         avg_tokens_per_sec=("tokens_per_sec","mean"),
         avg_peak_vram_gb=("peak_vram_gb","mean"),
         instruction_pass_rate=("instruction_pass","mean"))
    .reset_index())
selection_summary["instruction_pass_rate"]*=100
selection_summary=selection_summary.round(2)

print("[모델별 비교 결과]")
display(selection_summary)
print("[구조화 출력 사례]")
display(compare_df.loc[compare_df["category"]=="구조화 출력",
    ["model","response","raw_json_valid","cleaned_json_valid"]])

[모델별 비교 결과]


,model,avg_latency_sec,avg_tokens_per_sec,avg_peak_vram_gb,instruction_pass_rate
0,Gemma-3-4B-IT,5.23,19.31,8.04,80.0
1,Qwen3-8B,2.39,30.48,15.28,80.0


[구조화 출력 사례]


,model,response,raw_json_valid,cleaned_json_valid
3,Gemma-3-4B-IT,"```json\n[\n {\n ""model"": ""Qwen"",\n ""de...",False,False
8,Qwen3-8B,"{""model"": ""Qwen"", ""developer"": ""Tongyi Lab, Al...",False,False


## STEP 11. 모델 평가 결과 CSV 저장
3회차 Agent는 2회차에서 선택한 모델을 사용한다는 흐름으로 연결합니다(실습 코드는 Qwen3-8B를 공통 예제로 사용).

In [ ]:
compare_path=RESULTS_DIR/"02_model_evaluation_results.csv"
summary_path=RESULTS_DIR/"02_model_selection_summary.csv"
compare_df.to_csv(compare_path,index=False,encoding="utf-8-sig")
selection_summary.to_csv(summary_path,index=False,encoding="utf-8-sig")
print("저장 완료:",compare_path,summary_path)
print("팀 과제: 응답 속도/VRAM/형식 준수 중 가장 중요한 기준과 모델 선택 이유 5문장 작성")

저장 완료: /workspace/results/02_model_evaluation_results.csv /workspace/results/02_model_selection_summary.csv
팀 과제: 응답 속도/VRAM/형식 준수 중 가장 중요한 기준과 모델 선택 이유 5문장 작성


## STEP 12. 제출 전 확인
- [ ] 두 모델의 5문항 추론이 실행되었는가?
- [ ] `Peak VRAM`이 0이 아닌가?
- [ ] `Instruction Pass`가 의미적 정확도가 아닌 형식 검사임을 이해했는가?
- [ ] `02_model_evaluation_results.csv`와 `02_model_selection_summary.csv`를 저장했는가?
- [ ] 다음 3회차에서 필요한 Tool Calling용 JSON과 연결해서 설명할 수 있는가?